### Korea Coastal Flooding Information
*from Korea Hydrographic And Oceanographic Agency, Ministry of Oceans and Fisheries*

>이 데이터는 해양수산부 국립해양조사원에서 2009년~2020년까지 제작된 100년빈도 규모의 연안 인접지역 침수범위 예측정보를 제공하기 위해 수집한 데이터입니다.
검색 대상 시군구코드 등 값을 파라미터로 데이터를 조회할 수 있으며, 본 데이터의 주요 내용은 시도, 시군구명과 예측 침수값 및 침수시 공간정보 등 정보로 구성되어 있습니다.
본 데이터는 지방자치단체에서 침수 취약 지역의 방재 대책 수립 시 활용할 수 있으며, 재난관리 담당 부서가 안전 진단 및 예방 활동의 기초자료로 사용할 수 있습니다.
또한 침수 위험지구 관리와 연안 개발계획 수립 등 정책 수립 과정에서도 사용할 수 있습니다.

- **제공기관:** 해양수산부 국립해양조사원
- **관리부서명:** 해양예보과	관리부서 전화번호	051-400-4387
- **API 유형:** REST	데이터포맷	JSON+XML
- **End Point:** *https://apis.data.go.kr/1192136/waterlogged*
- **Service:**  */GetWaterloggedApiService*  [ 연안침수정보 : 우리나라 연안에 대한 조위별 침수 정보 제공 서비스 ]
- **침수값:** flodVlCn (Meter)

In [1]:
%useLatestDescriptors

%use dataframe
%use kandy

In [2]:
@file:DependsOn("org.json:json:20250107")
@file:DependsOn("org.xerial:sqlite-jdbc:3.49.1.0")
@file:DependsOn("ch.qos.logback:logback-classic:1.5.12")

In [3]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

**시군구 코드 정보**
- 출처 : *https://www.data.go.kr/cmm/cmm/fileDownload.do?atchFileId=FILE_000000003661213&fileDetailSn=1*
- **[오픈API 활용가이드_연안 침수 정보]** 내용중에서 **[연안 침수 정보 대상 시군구 코드]** 사용

In [4]:
val url_code = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/KotlinNotebooks/data/sggCode.csv"
val df_code = DataFrame.readCsv(url_code, ',')

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory


In [5]:
import java.sql.DriverManager

Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection(configData.SQLITE_DB.jdbcURL)
val dbPath = "jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite"


In [8]:
val tableName = "SggCode"

connection.use{ conn ->
    val sql = """INSERT INTO ${tableName} (sgg_code, sgg_name, sd_name) VALUES (?,?,? )""".trimIndent()
    df_code.select("시군구코드", "시군구명", "시도명").forEach { it ->
        try {
            conn.prepareStatement(sql)?.use { preparedStatement ->
                preparedStatement.setString(1, it["시군구코드"].toString())
                preparedStatement.setString(2, it["시군구명"].toString())
                preparedStatement.setString(3, it["시도명"].toString())
                preparedStatement.executeUpdate()
            }
        } catch (e: Exception){
            println(e.localizedMessage)
        }
    }

}

In [8]:
val sqlStmt = "SELECT * FROM SggCode"
val df_code = DataFrame.readSqlQuery(connection, sqlStmt)

In [40]:
val codeList = df_code.sgg_code.toList()

val numOfRows = 300

val allDataFrames = mutableListOf<DataFrame<*>>()

codeList.forEachIndexed { index, it ->



        val baseUrl = "${configData.WATER_LOGGED.endPoint}/${configData.WATER_LOGGED.subPath}" +
                "?serviceKey=${configData.WATER_LOGGED.apikey}&type=json&sggCd=${it}&numOfRows=${numOfRows}"

        val firstUrl = "${baseUrl}&pageNo=1"

        val df_first = DataFrame.readJson(firstUrl)
        val data = df_first["body"]["items"]["item"].first() as DataFrame<*>

        val totalCount = (df_first["body"]["totalCount"][0] as Number).toInt()
        val totalPages = ceil(totalCount.toDouble() / numOfRows).toInt()
        //   println("ssgNm:${it}, 시군구:${data[0][0]}/${data[0][1]}, 총 데이터 개수: $totalCount, 전체 페이지 수: $totalPages")

        val dataFrames = mutableListOf<DataFrame<*>>()

        dataFrames.add(data)

        for (page in 2..totalPages) {
            val url = "$baseUrl&pageNo=$page"
            val df_page = DataFrame.readJson(url)
            val data = df_page["body"]["items"]["item"].first() as DataFrame<*>
            dataFrames.add(data)
        }

        val df_SggAll = dataFrames.concat()

        allDataFrames.add(df_SggAll)



}

In [41]:
val df = allDataFrames.concat()

In [43]:
val tableNmCoastalFloodingInfo = "CoastalFloodingGeoInfo"

connection.use{ conn ->
    val sql = """INSERT INTO ${tableNmCoastalFloodingInfo} (ctpvNm, sggNm, flodVlCn, geom) VALUES (?,?,?,?)""".trimIndent()
    df.select("ctpvNm", "sggNm", "flodVlCn", "geom").forEach { it ->
        try {
            conn.prepareStatement(sql)?.use { preparedStatement ->
                preparedStatement.setString(1, it["ctpvNm"].toString())
                preparedStatement.setString(2, it["sggNm"].toString())
                preparedStatement.setString(3, it["flodVlCn"].toString())
                preparedStatement.setString(4, it["geom"].toString())
                preparedStatement.executeUpdate()
            }
        } catch (e: Exception){
            println(e.localizedMessage)
        }
    }

}

In [10]:
val sqlStmt = "SELECT * FROM CoastalFloodingGeoInfo "
val df_CoastalFlooding = DataFrame.readSqlQuery(connection, sqlStmt)

In [11]:
val countsDf = df_CoastalFlooding.groupBy { flodVlCn }
    .count()
    .sortByDesc("flodVlCn")

In [48]:
val joinedDf = df_CoastalFlooding.join(df_code) { (ctpvNm match right.sd_name) and (sggNm match right.sgg_name) }
    .add("alertLevel") {
        when (flodVlCn.trim()) {
            "0.0-0.5" -> 1
            "0.5-1.0" -> 2
            "1.0-1.5" -> 3
            "1.5-2.0" -> 4
            "2.0-2.5" -> 7
            "2.5-3.0" -> 7
            "2.0-3.0" -> 7
            "3.0" -> 8
            else -> 0
        }
    }

**WKT(Well-Known Text) 형식의 MULTIPOLYGON 문자열 리스트를을 GeoJSON 형식으로 변환하는 Kotlin 코드**

In [49]:
// 1. 각 WKT에서 "MULTIPOLYGON" 키워드를 제거하고 가장 겉의 괄호 1쌍을 제거
val extractedPolygons = joinedDf.geom.map { wkt ->
    wkt.trim()
        .replace(Regex("""^MULTIPOLYGON\s*""", RegexOption.IGNORE_CASE), "")
        .removePrefix("(")
        .removeSuffix(")")
        .trim()
}

In [50]:
fun parseWktToLatLngList(wktString: String): List<Map<String, Double>> {
    // 1. 괄호 제거 및 공백 정리
    val cleaned = wktString.replace("(", "").replace(")", "").trim()

    // 2. 쉼표(,)를 기준으로 각 좌표 쌍 분리
    val coordinatePairs = cleaned.split(",")

    // 3. 각 쌍을 공백으로 분리하여 LatLng 객체로 변환
    return coordinatePairs.mapNotNull { pair ->
        val parts = pair.trim().split("\\s+".toRegex())
        if (parts.size == 2) {
            val lng = parts[0].toDoubleOrNull()
            val lat = parts[1].toDoubleOrNull()

            if (lat != null && lng != null) {
                mapOf("lat" to lat, "lng" to lng) // 구글 맵 포맷인 (위도, 경도) 순서로 생성
                //  "{lat:${lat}, lng:${lng}}"
            } else null
        } else null
    }
}

In [ ]:
val latlngMapList = extractedPolygons.map { poligonString ->
    parseWktToLatLngList(poligonString)
}

- **Local Server End Point:** *http://192.168.35.107:7788*
- **Service:**  */khoa/coastal_flooding_info*

In [2]:
fun loadCoastalFlooding():List<DataFrame<*>>{
    var page = 1
    val size = 1000
    var currentCnt = 0
    val allDataFrames = mutableListOf<DataFrame<*>>()
    val baseUrl = "http://192.168.35.107:7788/khoa/coastal_flooding_info?size=${size}"
    do {
        val url = "${baseUrl}&page=${page}"
        val df = DataFrame.readJson(url)
        allDataFrames.add(df)
        currentCnt = df.count()
        page = page + 1
    } while (currentCnt == size)

    return allDataFrames
}

In [3]:
val result = loadCoastalFlooding().concat()

In [4]:
result.describe()

name,type,count,unique,nulls,top,freq,min,p25,median,p75,max
flodVlCn,String,59479,6,0,0.5-1.0,15076,0.0-0.5,0.5-1.0,1.0-1.5,1.5-2.0,3.0
grade,String,59479,6,0,B,15076,A,B,C,D,F
geom,String,59479,59479,0,MULTIPOLYGON(((125.39590418233408 34....,1,MULTIPOLYGON(((125.39590418233408 34....,MULTIPOLYGON(((126.51446144527098 34....,MULTIPOLYGON(((127.41296164663615 34....,MULTIPOLYGON(((128.4900620423059 35.0...,MULTIPOLYGON(((129.49101948828513 35....


In [5]:
result.groupBy{grade}.aggregate {
    flodVlCn.max() into "flodVlCn"
    count() into "count"
}


grade,flodVlCn,count
A,0.0-0.5,10596
B,0.5-1.0,15076
C,1.0-1.5,14362
D,1.5-2.0,10494
E,2.0-3.0,6628
F,3.0,2323
